# Модуль 1 · Лекція 4 — Функції, замикання, декоратори, генератори

👋 Фінальна (і найпотужніша) лекція основ. Функції — це те, з чого будується весь серйозний код.
Тут ми пройдемо шлях **від найпростішої функції до тем, якими «мучать» на співбесідах**:
LEGB, замикання, декоратори, генератори, менеджери контексту та GIL.

**Передумова:** [Лекція 3 — рядки та колекції](lektsiya-3-ryadky-ta-kolektsiyi.ipynb)

### Що ви вмітимете після лекції
- писати функції з різними видами аргументів (`*args`, `**kwargs`);
- розуміти **область видимості** (LEGB), `global`/`nonlocal`;
- створювати **замикання** та **декоратори**;
- писати **генератори** (`yield`) та розуміти ітератори;
- користуватися менеджерами контексту (`with`);
- пояснити, що таке **GIL** і чим `threading` відрізняється від `multiprocessing`.

> 📌 Позначка **🔎 Важливо знати** — теми, які майже завжди питають на технічних співбесідах.

# Частина 1. Основи функцій

## 1.1. Що таке функція й навіщо вона

**Функція** — це іменований блок коду, який можна **повторно викликати**. Замість того щоб
копіювати ті самі рядки, ви пишете їх раз у функції й далі просто **викликаєте** її за іменем.

Оголошують функцію словом `def`:
```
def ім'я_функції(параметри):
    тіло функції (з відступом)
    return результат
```

In [ ]:
# Оголошуємо функцію:
def greet():
    print("Привіт!")
    print("Ласкаво просимо")

# Саме оголошення нічого не друкує. Треба ВИКЛИКАТИ функцію — з дужками:
greet()
greet()   # можна викликати скільки завгодно разів

## 1.2. Параметри, аргументи та `return`

- **Параметр** — змінна в дужках при оголошенні (вхідні дані функції).
- **Аргумент** — конкретне значення, яке ви передаєте при виклику.
- **`return`** — повертає результат назовні. Без `return` функція повертає `None`.

In [ ]:
def square(number):        # number — параметр
    return number * number

result = square(5)         # 5 — аргумент; результат записуємо у змінну
print(result)              # 25
print(square(10))          # 100

# Функція може приймати кілька параметрів:
def add(a, b):
    return a + b

print(add(2, 3))           # 5

> 🧠 Різниця між `return` і `print`: `print` просто **виводить** на екран, а `return`
> **повертає значення**, яке можна далі використати (зберегти у змінну, передати далі).

In [ ]:
def with_return(x):
    return x * 2

def with_print(x):
    print(x * 2)           # тільки друкує, нічого не повертає

a = with_return(10)        # a = 20 (значення повернулось)
b = with_print(10)         # друкує 20, але b = None
print("a =", a)
print("b =", b)

## 1.3. Значення за замовчуванням

Параметру можна задати **значення за замовчуванням** — воно використається, якщо аргумент
не передали.

In [ ]:
def greet(name, greeting="Привіт"):
    return f"{greeting}, {name}!"

print(greet("Олено"))                 # Привіт, Олено!  (greeting за замовчуванням)
print(greet("Іване", "Вітаю"))        # Вітаю, Іване!   (передали своє)

## 1.4. Позиційні та іменовані аргументи

- **Позиційні** — зіставляються з параметрами **за порядком**.
- **Іменовані (keyword)** — передаються за іменем `параметр=значення`; **порядок неважливий**.

In [ ]:
def create_user(name, age, city="Львів"):
    return f"{name}, {age} р., {city}"

print(create_user("Іван", 30))                 # позиційні
print(create_user("Марія", 25, city="Київ"))   # іменований аргумент
print(create_user(age=40, name="Олег"))        # порядок неважливий для іменованих

## 1.5. 🔎 Важливо знати: `*args` та `**kwargs`

Іноді ми не знаємо заздалегідь, скільки аргументів передадуть. Тоді:
- **`*args`** збирає **зайві позиційні** аргументи у **кортеж** (`tuple`);
- **`**kwargs`** збирає **зайві іменовані** аргументи у **словник** (`dict`).

Імена `args`/`kwargs` — лише конвенція; головне — символи `*` і `**`.

In [ ]:
def demo(*args, **kwargs):
    print("args   (кортеж):", args)
    print("kwargs (словник):", kwargs)

demo(1, 2, 3, name="Аня", age=25)

In [ ]:
# Практичний приклад: сума будь-якої кількості чисел
def total(*numbers):
    return sum(numbers)

print(total(1, 2))            # 3
print(total(1, 2, 3, 4, 5))   # 15

**Зворотний бік — розпакування при виклику.** Ті самі `*` і `**` при виклику **розкладають**
список/словник на окремі аргументи:

In [ ]:
def point(x, y, z):
    return (x, y, z)

coords = [1, 2, 3]
print(point(*coords))       # * розпакує список у позиційні -> point(1, 2, 3)

params = {"x": 10, "y": 20, "z": 30}
print(point(**params))      # ** розпакує словник у іменовані -> point(x=10, y=20, z=30)

## 1.6. 🔎 Важливо знати: пастка змінюваного аргументу за замовчуванням

Одна з **найвідоміших пасток Python** (і улюблене питання співбесід). Значення за замовчуванням
обчислюється **лише один раз** — коли функція оголошується, а **не** при кожному виклику.
Якщо це змінюваний об'єкт (`list`, `dict`), він **зберігається між викликами**:

In [ ]:
# ❌ ПОГАНО: список створюється ОДИН раз і "живе" між викликами
def add_item(item, basket=[]):
    basket.append(item)
    return basket

print(add_item("яблуко"))     # ['яблуко']
print(add_item("банан"))      # ['яблуко', 'банан'] — сюрприз! той самий список

In [ ]:
# ✅ ДОБРЕ: за замовчуванням None, а список створюємо всередині
def add_item(item, basket=None):
    if basket is None:
        basket = []           # новий список при КОЖНОМУ виклику
    basket.append(item)
    return basket

print(add_item("яблуко"))     # ['яблуко']
print(add_item("банан"))      # ['банан'] — тепер правильно

# Частина 2. Функції — це теж об'єкти

## 2.1. Функції першого класу

У Python функція — **звичайний об'єкт**. Її можна: покласти у змінну, передати в іншу функцію
як аргумент, повернути з функції. Це фундамент декораторів (далі).

In [ ]:
def shout(text):
    return text.upper() + "!"

# Кладемо функцію у змінну (БЕЗ дужок — не викликаємо, а посилаємось):
f = shout
print(f("привіт"))       # ПРИВІТ!

In [ ]:
# Функція, що приймає ІНШУ функцію як аргумент (higher-order function):
def apply_twice(func, value):
    return func(func(value))

def add_three(x):
    return x + 3

print(apply_twice(add_three, 10))    # 16  (10 -> 13 -> 16)

## 2.2. `lambda` — анонімна функція

`lambda` — це коротка функція **з одного виразу**, без імені. Зручна, коли функція потрібна
«на один раз» — наприклад, передати в іншу функцію.

In [ ]:
# Звичайна функція:
def add(a, b):
    return a + b

# Те саме як lambda:
add_lambda = lambda a, b: a + b

print(add(2, 3))          # 5
print(add_lambda(2, 3))   # 5

# Типове застосування — передати "на льоту":
def apply(func, value):
    return func(value)

print(apply(lambda x: x * 10, 5))   # 50

## 2.3. `map` та `filter`

- **`map(функція, послідовність)`** — застосовує функцію до **кожного** елемента.
- **`filter(функція, послідовність)`** — лишає лише ті елементи, для яких функція дала `True`.

Обидві повертають «лінивий» об'єкт, тож для перегляду обгортаємо в `list()`.

In [ ]:
nums = [1, 2, 3, 4, 5]

# map: піднести кожне число до квадрата
squares = list(map(lambda x: x ** 2, nums))
print("квадрати:", squares)          # [1, 4, 9, 16, 25]

# filter: лишити тільки парні
evens = list(filter(lambda x: x % 2 == 0, nums))
print("парні:", evens)               # [2, 4]

> 💡 Часто те саме читабельніше через comprehension (Лекція 3):
> `[x ** 2 for x in nums]` замість `map`, `[x for x in nums if x % 2 == 0]` замість `filter`.

# Частина 3. 🔎 Важливо знати: область видимості (LEGB)

Коли Python зустрічає ім'я змінної, він шукає його у **чотирьох** просторах саме в такому
порядку (перше знайдене — виграє). Це правило називається **LEGB**:

1. **L — Local:** усередині поточної функції.
2. **E — Enclosing:** у зовнішній функції (якщо є вкладеність).
3. **G — Global:** на рівні модуля (файлу).
4. **B — Built-in:** вбудовані імена (`len`, `print`, `range`…).

In [ ]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print("inner бачить:", x)    # local
    inner()
    print("outer бачить:", x)        # enclosing

outer()
print("модуль бачить:", x)           # global

## 3.1. `global` та `nonlocal`

За замовчуванням присвоєння `x = ...` усередині функції **створює нову локальну** змінну.
Щоб змінювати зовнішню:
- **`global x`** — працювати зі змінною рівня модуля;
- **`nonlocal x`** — працювати зі змінною охоплюючої функції.

In [ ]:
counter = 0

def increment():
    global counter        # без цього рядка створилась би локальна змінна
    counter += 1

increment()
increment()
print("counter:", counter)   # 2

In [ ]:
# nonlocal — змінюємо змінну ОХОПЛЮЮЧОЇ функції:
def make_counter():
    count = 0
    def step():
        nonlocal count
        count += 1
        return count
    return step

c = make_counter()
print(c(), c(), c())        # 1 2 3

### 🔎 Важливо знати: `UnboundLocalError`
Якщо всередині функції є **присвоєння** змінної, Python вважає її **локальною для всієї функції** —
навіть у рядках **до** присвоєння. Спроба прочитати її раніше дає помилку:

In [ ]:
value = 10

def broken():
    value = value + 1     # value вважається ЛОКАЛЬНОЮ (бо є присвоєння),
    return value          # але праворуч її ще не існує -> помилка

try:
    broken()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

# Частина 4. 🔎 Важливо знати: замикання (closures)

**Замикання** — це вкладена функція, яка **пам'ятає змінні** зі своєї охоплюючої функції навіть
**після того**, як зовнішня функція завершилась. Саме на замиканнях побудовані декоратори.

In [ ]:
def multiplier(factor):
    def multiply(number):
        return number * factor    # factor "замкнено" із зовнішньої функції
    return multiply

double = multiplier(2)            # factor = 2 "запам'ятався"
triple = multiplier(3)            # factor = 3

print(double(10))    # 20
print(triple(10))    # 30

### 🔎 Важливо знати: пастка пізнього зв'язування (late binding)

Класичне питання. Змінна із замикання «читається» **в момент виклику**, а не створення. Тому
всі лямбди в циклі бачать **останнє** значення:

In [ ]:
# ❌ Несподіванка: усі функції повертають 2
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])          # [2, 2, 2] — усі бачать останнє i

# ✅ Виправлення: зафіксувати значення через аргумент за замовчуванням
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])          # [0, 1, 2]

# Частина 5. 🔎 Важливо знати: декоратори

**Декоратор** — функція, що приймає іншу функцію й повертає **нову** функцію з доданою поведінкою
(логування, кешування, перевірка доступу) — **не змінюючи** код оригіналу. Синтаксис `@decorator`
над функцією — це просто скорочення для `func = decorator(func)`.

Побудуймо крок за кроком.

In [ ]:
# Крок 1: декоратор вручну
def with_logging(func):
    def wrapper(*args, **kwargs):
        print(f"-> викликаю {func.__name__}")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} повернула {result}")
        return result
    return wrapper

def add(a, b):
    return a + b

add = with_logging(add)   # "обгортаємо" вручну
add(2, 3)

In [ ]:
# Крок 2: те саме через синтаксис @ — читабельніше
def with_logging(func):
    def wrapper(*args, **kwargs):
        print(f"-> {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"<- {result}")
        return result
    return wrapper

@with_logging          # еквівалент: multiply = with_logging(multiply)
def multiply(a, b):
    return a * b

multiply(4, 5)

### `functools.wraps` — навіщо

Коли ми обгортаємо функцію, вона «втрачає» своє ім'я й документацію (стає `wrapper`).
`functools.wraps` копіює ці метадані з оригіналу — **завжди** додавайте його у свої декоратори:

In [ ]:
import functools

def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def greet():
    """Вітає користувача."""
    return "Привіт"

print("Без wraps:", greet.__name__, "|", greet.__doc__)   # 'wrapper' — метадані загубились!

In [ ]:
import functools

def good_decorator(func):
    @functools.wraps(func)         # зберігає __name__, __doc__ оригіналу
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def greet2():
    """Вітає користувача."""
    return "Привіт"

print("З wraps:", greet2.__name__, "|", greet2.__doc__)

### Готовий декоратор зі стандартної бібліотеки: `lru_cache`

`functools.lru_cache` **запам'ятовує** результати (мемоізація) — повільна рекурсивна функція
стає миттєвою:

In [ ]:
import functools

@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(50))            # миттєво, попри рекурсію
print(fib.cache_info())   # статистика кешу

# Частина 6. 🔎 Важливо знати: ітератори та генератори

Це фундамент, на якому працює `for`, і **дуже** часте питання співбесід.

## 6.1. Iterable vs Iterator

- **Iterable (ітерабельний)** — об'єкт, який **можна перебирати** (`list`, `str`, `dict`, `range`).
  Має метод `__iter__()`, що повертає ітератор.
- **Iterator (ітератор)** — об'єкт, що **видає елементи по одному** через `__next__()`;
  коли елементи закінчились — кидає `StopIteration`.

**Цикл `for` під капотом:** бере `iter(об'єкт)` → отримує ітератор → у циклі кличе `next()`,
поки не впіймає `StopIteration`.

In [ ]:
nums = [10, 20, 30]

it = iter(nums)          # зі списку (iterable) отримали ітератор
print(next(it))          # 10
print(next(it))          # 20
print(next(it))          # 30

try:
    next(it)             # елементи скінчились
except StopIteration:
    print("StopIteration — елементи закінчились")

In [ ]:
# Приблизно ОТАК "for" працює зсередини:
def my_for(iterable, action):
    iterator = iter(iterable)
    while True:
        try:
            item = next(iterator)
        except StopIteration:
            break
        action(item)

my_for(["a", "b", "c"], lambda x: print("елемент:", x))

## 6.2. Генератори (`yield`)

**Генератор** — найпростіший спосіб створити ітератор. Це функція, яка замість `return`
використовує **`yield`**. Виклик такої функції не виконує тіло одразу, а повертає
**генератор-об'єкт**.

Головна ідея — **лінива (lazy) обробка**: значення обчислюються по одному, **лише коли їх
запитують**. Тому генератор майже не займає пам'ять і може бути навіть **нескінченним**.

In [ ]:
def countdown(n):
    print("  генератор стартував")
    while n > 0:
        yield n           # "віддає" значення і ЗАСТИГАЄ тут до наступного next()
        n -= 1

gen = countdown(3)        # тіло ще НЕ виконувалось!
print("створили генератор:", gen)
print(next(gen))          # аж тепер друкує "стартував" і повертає 3
print(next(gen))          # 2
print(next(gen))          # 1

**Генераторний вираз** — як list comprehension, але в круглих дужках і **лінивий**.
Порівняймо, скільки пам'яті займають список і генератор:

In [ ]:
import sys

list_version = [x * x for x in range(100_000)]   # усе одразу в пам'яті
gen_version = (x * x for x in range(100_000))     # лінивий — майже нічого не займає

print("список займає байтів:   ", sys.getsizeof(list_version))
print("генератор займає байтів:", sys.getsizeof(gen_version))

> ⚠️ Генератор **вичерпується** — по ньому можна пройти лише **один раз**:

In [ ]:
gen = (x for x in range(3))
print("перший прохід:", list(gen))   # [0, 1, 2]
print("другий прохід:", list(gen))   # [] — вже порожній!

## 6.3. `itertools` — готові інструменти для ітерування

Стандартний модуль зі швидкими «будівельними блоками» для роботи з послідовностями
(у т.ч. нескінченними).

In [ ]:
import itertools

# islice — "зріз" для будь-якого ітератора (навіть нескінченного count):
print(list(itertools.islice(itertools.count(10, 5), 4)))   # [10, 15, 20, 25]

# chain — послідовно об'єднати кілька послідовностей:
print(list(itertools.chain([1, 2], [3, 4], [5])))          # [1, 2, 3, 4, 5]

# combinations — усі можливі пари:
print(list(itertools.combinations(["A", "B", "C"], 2)))    # [('A','B'), ('A','C'), ('B','C')]

# Частина 7. Менеджери контексту (`with`)

`with` гарантує, що ресурс буде **коректно звільнено** (файл закрито, з'єднання розірвано) —
навіть якщо всередині блоку станеться помилка. Найчастіший приклад — робота з файлами.

In [ ]:
import os

# Файл сам закриється після блоку with — навіть при помилці всередині:
with open("demo.txt", "w", encoding="utf-8") as f:
    f.write("Привіт із with!")

with open("demo.txt", encoding="utf-8") as f:
    print(f.read())

os.remove("demo.txt")   # прибираємо тестовий файл

### 🔎 Важливо знати: як зробити власний менеджер контексту

Об'єкт-менеджер реалізує два методи: **`__enter__`** (виконується на вході в блок) і
**`__exit__`** (на виході — навіть при винятку; тут прибирання). Простіший спосіб — через
`contextlib` з `yield`:

In [ ]:
from contextlib import contextmanager

@contextmanager
def tag(name):
    print(f"<{name}>")
    yield                 # усе ДО yield — це вхід (__enter__), усе ПІСЛЯ — вихід (__exit__)
    print(f"</{name}>")

with tag("b"):
    print("жирний текст")

# Частина 8. 🔎 Важливо знати: GIL, потоки та процеси (оглядово)

Тема поглиблена, але про **GIL** питають майже на кожній Python-співбесіді. Ось короткий,
але правильний огляд.

### Concurrency vs Parallelism
- **Concurrency (конкурентність)** — задачі *чергуються* й прогресують «майже одночасно»
  (швидке перемикання). Добре для **I/O-bound** задач (мережа, диск — там багато очікування).
- **Parallelism (паралелізм)** — задачі *справді* виконуються одночасно на різних ядрах CPU.
  Потрібно для **CPU-bound** задач (важкі обчислення).

### GIL (Global Interpreter Lock)
У CPython є глобальне блокування, через яке **в один момент часу байткод виконує лише один потік**.
Наслідок: `threading` **не прискорює** обчислення на CPU, але **допомагає** при очікуванні I/O
(поки один потік чекає мережу, інший працює).

### Три підходи в Python

| Підхід | Модуль | Для чого | GIL |
|---|---|---|---|
| Багатопоточність | `threading` | I/O-bound | ⚠️ заважає паралелізму на CPU |
| Багатопроцесність | `multiprocessing` | CPU-bound | обходить GIL (окремі процеси) |
| Асинхронність | `asyncio` | I/O-bound, тисячі з'єднань | один потік, кооперативно |

In [ ]:
# threading: кілька потоків для I/O-подібної задачі (очікування).
import threading, time

def worker(n):
    time.sleep(0.2)               # імітація очікування I/O
    print(f"  потік {n} завершив")

threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()                      # чекаємо завершення всіх
print("усі потоки завершились")

> 📎 `asyncio` (async/await) — окрема велика тема; на цьому рівні достатньо розуміти,
> **коли** що застосовувати: `threading`/`asyncio` для I/O, `multiprocessing` для CPU.
> Глибше — за посиланнями в кінці лекції.

# ✅ Підсумок лекції

- **Функції:** `def`, параметри/аргументи, `return`, значення за замовчуванням.
- **`*args`/`**kwargs`** — довільна кількість аргументів; `*`/`**` при виклику — розпакування.
- **Пастка:** не використовуйте змінюваний об'єкт (`[]`, `{}`) як значення за замовчуванням — беріть `None`.
- **Функції — об'єкти:** можна передавати й повертати; `lambda`, `map`, `filter`.
- **LEGB:** Local → Enclosing → Global → Built-in; `global`/`nonlocal`; `UnboundLocalError`.
- **Замикання** пам'ятають зовнішні змінні; обережно з пізнім зв'язуванням у циклі.
- **Декоратори:** обгортка `@decorator`; завжди `functools.wraps`; готовий `lru_cache`.
- **Ітератори/генератори:** `iter`/`next`/`StopIteration`; `yield` — лінива обробка; вичерпуються.
- **`with`** — гарантоване звільнення ресурсів.
- **GIL:** один потік байткоду в CPython; `threading`/`asyncio` для I/O, `multiprocessing` для CPU.

### 📝 Домашнє завдання
Виконайте вправи в окремому зошиті: [domashnie-zavdannia-lektsiya-4.ipynb](domashnie-zavdannia-lektsiya-4.ipynb)

### 🎉 Вітаємо — це кінець Модуля 1 (основи Python)!
Ви пройшли шлях від `print("Привіт")` до генераторів, декораторів і GIL. Тепер ви готові до
**Модуля 2 — Об'єктно-орієнтоване програмування (ООП)**, де застосуємо ці знання в реальній
архітектурі проєкту.

### 📚 Хочу знати більше
- Функції (туторіал, укр.): <https://docs.python.org/uk/3/tutorial/controlflow.html#defining-functions>
- Область видимості (LEGB): <https://realpython.com/python-scope-legb-rule/>
- Декоратори: <https://realpython.com/primer-on-python-decorators/>
- Генератори: <https://realpython.com/introduction-to-python-generators/>
- `functools` / `contextlib`: <https://docs.python.org/3/library/functools.html>
- GIL, потоки, процеси: <https://realpython.com/python-gil/>
- Async IO: <https://realpython.com/async-io-python/>